# 03 — Gaussian Splatting

3D Gaussian Splattingは、sceneを多数の3D Gaussianで表現します。描画画像と撮影画像の差から、Gaussianの位置・形・透明度・色を最適化します。

In [ ]:
import json
import os
import sys
from pathlib import Path

venv_bin = str(Path(sys.executable).parent)
os.environ["PATH"] = venv_bin + os.pathsep + os.environ["PATH"]

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from IPython.display import Image as IPythonImage

from gs_tutorial.config import load_config
from gs_tutorial.dataset import load_colmap_text, scene_summary

cwd = Path.cwd().resolve()
repo = cwd.parent if (cwd.parent / "pyproject.toml").is_file() else cwd
config_path = repo / "configs/video.yaml"  # Use images.yaml for multi-view image input
cfg = load_config(config_path)
processed = repo / cfg.project_dir / "colmap" / "processed"
scene = load_colmap_text(processed / "sparse_txt")
scene_summary(scene)

## 1. Gaussianの初期化

COLMAPの各3D点を一つのGaussianとして初期化します。主なparameterは次のとおりです。

- `means`: 中心位置
- `scales`, `quats`: 大きさと向き
- `opacities`: 不透明度
- `sh0`, `shN`: 視線方向を含む色

出力されたshapeを見て、COLMAP点数と初期Gaussian数が一致することを確認します。

In [ ]:
import torch

from gs_tutorial.training import initialize_gaussians

splats = initialize_gaussians(scene, cfg.training.sh_degree)
print({name: tuple(value.shape) for name, value in splats.items()})
{
    "initial_gaussians": len(splats["means"]),
    "opacity_mean": torch.sigmoid(splats["opacities"]).mean().item(),
    "scale_median": torch.exp(splats["scales"]).median().item(),
}

## 2. Gaussianの最適化

各stepで一つのcameraから描画画像を作り、撮影画像との差をGaussian parameterへ逆伝播します。

lossは色差を測る`L1`と構造を測る`1 − SSIM`の加重和です。

学習中はGaussianを複製・分割・削除します。`Gaussian count`とlossの変化、preview左の撮影画像と右の描画画像を確認します。

In [ ]:
import ipywidgets as widgets

from gs_tutorial.training import train

training_dir = repo / cfg.project_dir / "training"
status_widget = widgets.HTML()
preview_widget = widgets.Image(format="jpeg", width=720)
display(status_widget, preview_widget)


def update_training(event):
    if "loss" in event:
        status_widget.value = (
            f"<b>step {event['step']}</b> loss={event['loss']:.5f}, "
            f"SSIM={event['ssim']:.4f}, PSNR={event['psnr']:.2f} dB, "
            f"Gaussians={event['gaussians']:,}"
        )
    if "preview" in event:
        preview_widget.value = Path(event["preview"]).read_bytes()

In [ ]:
metrics_path = training_dir / "metrics.json"
final_ply = training_dir / "point_cloud" / "final.ply"
has_training_files = training_dir.is_dir() and any(training_dir.iterdir())
if final_ply.is_file() and metrics_path.is_file():
    print("Reusing completed training outputs; training will not run again")
    history = json.loads(metrics_path.read_text(encoding="utf-8"))
elif has_training_files:
    raise RuntimeError(
        f"Training directory {training_dir} exists but is incomplete. Remove it or run with --overwrite to start a new training. "
    )
else:
    splats, history = train(
        scene, processed / "images", training_dir, cfg.training, update_training
    )
print("history rows:", len(history))

In [ ]:
steps = [row["step"] for row in history]
figure, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(steps, [row["loss"] for row in history])
axes[0].set(title="Loss", xlabel="step")
axes[1].plot(steps, [row["psnr"] for row in history])
axes[1].set(title="PSNR", xlabel="step")
axes[2].plot(steps, [row["gaussians"] for row in history])
axes[2].set(title="Gaussian count", xlabel="step")
figure.tight_layout()
previews = sorted((training_dir / "previews").glob("step_*.jpg"))
if previews:
    display(IPythonImage(filename=str(previews[-1]), width=960))

## 3. 撮影画像 / 描画画像の評価

学習時と同じcameraから描画画像を作り、撮影画像と比較します。

- `PSNR`: 高いほど色が近い
- `SSIM`: 1に近いほど構造が近い
- `L1`: 低いほど色差が小さい

best/worst viewを見て、遮蔽・反射・撮影範囲端で品質が落ちていないか確認します。

In [ ]:
from gs_tutorial.evaluation import render_registered_views

checkpoint = training_dir / "checkpoints" / f"step_{cfg.training.max_steps:06d}.pt"
evaluation_dir = repo / cfg.project_dir / "evaluation"
evaluation_metrics = evaluation_dir / "metrics.json"
evaluation_status = widgets.HTML()
display(evaluation_status)


def update_evaluation(event):
    evaluation_status.value = (
        f"view {event['index']}/{event['total']}: {event['name']} — "
        f"PSNR {event['psnr']:.2f}, SSIM {event['ssim']:.4f}"
    )


if evaluation_metrics.is_file():
    print("Reusing completed evaluation outputs")
    evaluation = json.loads(evaluation_metrics.read_text(encoding="utf-8"))
else:
    evaluation = render_registered_views(
        scene,
        processed / "images",
        checkpoint,
        evaluation_dir,
        cfg.training,
        callback=update_evaluation,
    )
evaluation["aggregate"]

In [ ]:
rows = evaluation["per_view"]
best = max(rows, key=lambda row: row["psnr"])
worst = min(rows, key=lambda row: row["psnr"])
print("best:", best)
print("worst:", worst)
display(IPythonImage(filename=str(evaluation_dir / "comparison" / best["name"]), width=960))
display(IPythonImage(filename=str(evaluation_dir / "comparison" / worst["name"]), width=960))
gallery_path = evaluation_dir / "index.html"
display(HTML(f'<a href="{gallery_path.as_uri()}" target="_blank">比較galleryを開く</a>'))

## 4. 再構成結果の可視化

ViserではGaussian、初期点群、撮影cameraの表示を個別に切り替え、camera frustumをclickして撮影viewへ移動できます。

`initial.ply`と`step_*.ply`を保存しているため、学習後も`Training step` sliderで形状の変化を追跡できます。

JupyterHubでは、このNotebookとは別のterminalで次を実行してviewerを起動します。

```bash
source scripts/activate.sh
gs-tutorial viewer \
    outputs/my_video/training/point_cloud/final.ply \
    --config configs/video.yaml \
    --port 8000
```

手元のPCからport 8000をforwardする方法は[README](../README.md#viewerをjupyterhubで開く)を参照してください。

In [ ]:
ply_dir = training_dir / "point_cloud"
snapshots = [ply_dir / "initial.ply", *sorted(ply_dir.glob("step_*.ply")), ply_dir / "final.ply"]
[(path.name, f"{path.stat().st_size / 1e6:.1f} MB") for path in snapshots if path.is_file()]

In [ ]:
print("Viewer input:", final_ply)
print("Start the viewer in a separate JupyterHub terminal; see the command above.")